In [ ]:
!pip install datasets


INFO: pip is looking at multiple versions of multiprocess to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.6/471.6 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 9.3 MB/s eta 0:00:00


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import Trainer, TrainingArguments
import torch
from datasets import load_dataset

# Load your dataset (replace with your CSV file)
data = pd.read_csv('peer_pressure_data.csv')  # Ensure you have 'text' and 'label' columns

# Check for unique labels in your dataset
print(data['label'].unique())

# If you have more than two labels, adjust num_labels accordingly
num_labels = len(data['label'].unique())

# Split the dataset into training and testing sets
train_texts, test_texts, train_labels, test_labels = train_test_split(
    data['text'].tolist(),
    data['label'].tolist(),
    test_size=0.2,
    random_state=42
)

# Load BERT tokenizer and model
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=num_labels)  # Adjust num_labels based on your data

# Tokenize the texts
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=512)
test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=512)

# Create a PyTorch dataset
class PeerPressureDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

# Create datasets
train_dataset = PeerPressureDataset(train_encodings, train_labels)
test_dataset = PeerPressureDataset(test_encodings, test_labels)

# Define training arguments
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

# Train the model
trainer.train()

# Save the model
model.save_pretrained('./peer_pressure_model')
tokenizer.save_pretrained('./peer_pressure_model')

# Example inference
def predict(text):
    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True, max_length=512)
    outputs = model(**inputs)
    predictions = torch.argmax(outputs.logits, dim=1)
    return predictions.item()

# Test the model with a sample text
sample_text = "Everyone is going to the party, and I feel I should too."
print("Predicted label:", predict(sample_text))

[0 1 2]


Step,Training Loss


Step,Training Loss


Predicted label: 1


In [ ]:
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import GPT2LMHeadModel, GPT2Tokenizer, Trainer, TrainingArguments
from datasets import Dataset

# Load your dataset (replace with your CSV file)
data = pd.read_csv('peer_pressure_data.csv')  # Ensure you have 'text' and 'label' columns
num_labels = len(data['label'].unique())
# Split the dataset into training and testing sets
train_texts, test_texts, train_labels, test_labels = train_test_split(
    data['text'].tolist(),
    data['label'].tolist(),
    test_size=0.2,
    random_state=42
)

# Load BERT tokenizer and model
bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=num_labels)  # Assuming binary classification

# Tokenize the texts
train_encodings = bert_tokenizer(train_texts, truncation=True, padding=True, max_length=512)
test_encodings = bert_tokenizer(test_texts, truncation=True, padding=True, max_length=512)

# Create datasets for Hugging Face
train_dataset = Dataset.from_dict({
    'input_ids': train_encodings['input_ids'],
    'attention_mask': train_encodings['attention_mask'],
    'labels': train_labels
})

test_dataset = Dataset.from_dict({
    'input_ids': test_encodings['input_ids'],
    'attention_mask': test_encodings['attention_mask'],
    'labels': test_labels
})

# Define training arguments for BERT
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
)

# Initialize Trainer for BERT
trainer = Trainer(
    model=bert_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

# Train the BERT model
trainer.train()

# Save the BERT model and tokenizer
bert_model.save_pretrained('./peer_pressure_model')
bert_tokenizer.save_pretrained('./peer_pressure_model')

# Load GPT-2 tokenizer and model for emotional support generation
gpt2_tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
gpt2_model = GPT2LMHeadModel.from_pretrained('gpt2')

# Inference function for BERT
def predict(text):
    inputs = bert_tokenizer(text, return_tensors='pt', truncation=True, padding=True, max_length=512)
    with torch.no_grad():
        outputs = bert_model(**inputs)
    predictions = torch.argmax(outputs.logits, dim=1)
    return predictions.item()

# Function to generate emotional support responses
def generate_emotional_support(user_input, label):
    prompt = f"User feels: '{user_input}'. Generate a supportive response considering this feeling. in 100 words"
    inputs = gpt2_tokenizer.encode(prompt, return_tensors='pt')
    outputs = gpt2_model.generate(inputs, max_length=100, num_return_sequences=1, pad_token_id=gpt2_tokenizer.eos_token_id)
    response = gpt2_tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response

# Test the model with a sample text and generate a supportive response
sample_text = "Everyone is going to the party, and I feel I should too."
label = predict(sample_text)

# Generate a response based on the label and input
supportive_response = generate_emotional_support(sample_text, label)
print("Predicted label:", label)
print("AI Generated Supportive Response:", supportive_response)


Step,Training Loss
10,1.265900
20,1.190600
30,1.026900
40,0.936600
50,0.749100
60,0.615800
70,0.532600
80,0.445400
90,0.338400
100,0.227300


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Predicted label: 1
AI Generated Supportive Response: User feels: 'Everyone is going to the party, and I feel I should too.'. Generate a supportive response considering this feeling.

The next step is to create a social media presence.

This is where you can create a
